# 04 — Layer-Freezing Ablation: How Many Layers Should You Unfreeze?

**Goal of this notebook**: `notebooks/05_transfer_learning_densenet.ipynb` unfreezes
the last 30 DenseNet layers for fine-tuning — a reasonable default, but not one we
actually tested against alternatives. This notebook runs a small systematic
experiment: train the same architecture with a few different numbers of unfrozen
layers, and see how validation accuracy actually responds.

**What this isolates**: freezing more layers means the model relies more on raw
ImageNet features; unfreezing more means more capacity to adapt to MRI specifically,
but more risk of overfitting given the dataset's small size. There's a real trade-off
here, not an obviously-correct answer — that's exactly why it's worth measuring rather
than assuming.

**Compute note**: this trains several models, so it takes noticeably longer to run
than the other research notebooks. Reduce `epochs` below or the list of values in
`layers_to_test` if you want a quicker pass first.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from src.data_utils import build_tf_dataset
from src.models import build_transfer_model, unfreeze_for_finetuning
from src.train import train_model

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PER_RUN = 10  # reduced from the main pipeline's 15 — this notebook trains several models

In [ ]:
split_metadata = pd.read_csv("../data/processed/metadata_split.csv")


def to_3channel(images, labels):
    return tf.repeat(images, repeats=3, axis=-1), labels


train_dataset = build_tf_dataset(
    split_metadata, "train", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True
).map(to_3channel)
val_dataset = build_tf_dataset(
    split_metadata, "val", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
).map(to_3channel)

## Run the ablation

**Intent**: same starting architecture, same data, same epoch budget for every run —
number of unfrozen layers is the only thing that changes between runs, so any
difference in results is attributable to that one variable.

In [ ]:
layers_to_test = [
    0,
    10,
    30,
    100,
]  # 0 = fully frozen base (Phase 1 only); 100 ≈ nearly all of DenseNet121

results = {}
for n_layers in layers_to_test:
    run_name = f"ablation_unfreeze_{n_layers}"
    print(f"\n=== Training with {n_layers} unfrozen layers ===")

    model = build_transfer_model(
        input_shape=(*IMG_SIZE, 3), num_classes=len(CLASS_NAMES), freeze_base=True
    )
    if n_layers > 0:
        model = unfreeze_for_finetuning(model, num_layers_to_unfreeze=n_layers)

    history = train_model(
        model,
        train_dataset,
        val_dataset,
        run_name=run_name,
        epochs=EPOCHS_PER_RUN,
        checkpoint_dir=Path("../models/saved_models/ablation_freezing"),
        patience=5,
    )
    results[n_layers] = {
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "final_train_accuracy": history.history["accuracy"][-1],
        "final_val_accuracy": history.history["val_accuracy"][-1],
    }

## Results table and plot

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = "unfrozen_layers"
results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df.index, results_df["best_val_accuracy"], marker="o", label="Best val accuracy")
plt.plot(
    results_df.index,
    results_df["final_train_accuracy"] - results_df["final_val_accuracy"],
    marker="s",
    label="Final train/val gap (overfitting)",
)
plt.xlabel("Number of unfrozen DenseNet layers")
plt.ylabel("Score")
plt.title("Effect of unfreezing depth on validation accuracy and overfitting")
plt.legend()
plt.show()

## Conclusion

Fill in after running: which number of unfrozen layers gave the best validation
accuracy? Did the train/val gap grow noticeably at the higher end (100 layers), 
suggesting overfitting from too much fine-tuning capacity relative to the dataset
size? If a different value than 30 wins here, that's worth going back and updating
`notebooks/05_transfer_learning_densenet.ipynb`'s default — this notebook's whole
purpose is to make that choice evidence-based rather than assumed.